In [1]:
%pip install folium pandas requests

  Using cached branca-0.8.2-py3-none-any.whl.metadata (1.7 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached xyzservices-2026.3.0-py3-none-any.whl.metadata (4.1 kB)
Using cached branca-0.8.2-py3-none-any.whl (26 kB)
Using cached jinja2-3.1.6-py3-none-any.whl (134 kB)
Using cached xyzservices-2026.3.0-py3-none-any.whl (94 kB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [14]:
import folium
import json
import requests
import time
import os
import pandas as pd
import re

# 1. Warna Pastel Macaron
warna_wilayah = {
    "pusat": "#f8a5c2",   
    "utara": "#63cdda",   
    "selatan": "#78e08f", 
    "timur": "#cd84f1",   
    "barat": "#f6b93b"    
}

# =================================================================
# FUNGSI AJAIB EKSTRAKTOR (UPGRADED - LEBIH BAR-BAR)
# =================================================================
def ekstrak_jarak_dari_ipynb(nama_file):
    hasil_jarak = {"selatan": 0, "timur": 0, "utara": 0, "barat": 0, "pusat": 0}
    
    if not os.path.exists(nama_file):
        print(f"⚠️ File {nama_file} ga ketemu.")
        return hasil_jarak
        
    try:
        with open(nama_file, 'r', encoding='utf-8') as f:
            notebook = json.load(f)
            
        semua_teks_output = ""
        for cell in notebook.get('cells', []):
            if cell.get('cell_type') == 'code':
                for output in cell.get('outputs', []):
                    # 1. Sedot dari hasil print() biasa (stream)
                    if 'text' in output:
                        teks = output['text']
                        semua_teks_output += "".join(teks) if isinstance(teks, list) else teks
                    
                    # 2. Sedot dari hasil return variabel (execute_result/display_data)
                    elif 'data' in output and 'text/plain' in output['data']:
                        teks = output['data']['text/plain']
                        semua_teks_output += "".join(teks) if isinstance(teks, list) else teks

        # Pola Regex dibikin lebih fleksibel (Nangkap spasi/karakter aneh di antara Wilayah dan Jarak)
        pola = r"Wilayah\s+([A-Za-z]+).*?Jarak:\s*([0-9.]+)"
        cocokan = re.findall(pola, semua_teks_output, re.IGNORECASE | re.DOTALL)
        
        for match in cocokan:
            wilayah = match[0].lower()
            jarak = float(match[1])
            if wilayah in hasil_jarak:
                # Update jarak, kalau nemu yang sama di cell bawah, bakal ketimpa (ambil yang paling update)
                hasil_jarak[wilayah] = jarak
                
    except Exception as e:
        print(f"Ada error pas baca {nama_file}: {e}")
        
    return hasil_jarak

def load_koordinat(filepath):
    koordinat_lokasi = {}
    try:
        df = pd.read_csv(filepath)
        for index, row in df.iterrows():
            nama_lokasi = row['Nama'].strip() 
            lat = float(row['Latitude'])
            lon = float(row['Longitude'])
            koordinat_lokasi[nama_lokasi] = [lat, lon]
    except Exception:
        pass
    return koordinat_lokasi

def get_real_route_osrm(coord_start, coord_end):
    lon1, lat1 = coord_start[1], coord_start[0]
    lon2, lat2 = coord_end[1], coord_end[0]
    
    url = f"http://router.project-osrm.org/route/v1/driving/{lon1},{lat1};{lon2},{lat2}?overview=full&geometries=geojson"
    try:
        response = requests.get(url)
        data = response.json()
        if data["code"] == "Ok":
            route_coords = data["routes"][0]["geometry"]["coordinates"]
            folium_route = [[coord[1], coord[0]] for coord in route_coords]
            distance = data["routes"][0]["distance"]
            return folium_route, distance
    except Exception:
        pass 
    return [coord_start, coord_end], 0

def buat_peta_rute():
    path_csv = '../data/koordinat_puskesmas.csv'
    koordinat_lokasi = load_koordinat(path_csv)
    
    if len(koordinat_lokasi) == 0:
        return None

    nama_gudang = "UPTD Gudang Farmasi Surabaya"
    titik_pusat = koordinat_lokasi.get(nama_gudang, list(koordinat_lokasi.values())[0])

    peta = folium.Map(location=titik_pusat, zoom_start=12, tiles="CartoDB positron", zoom_control=False)
    folium.plugins.LocateControl().add_to(peta)

    if nama_gudang in koordinat_lokasi:
        folium.Marker(
            location=titik_pusat,
            popup="<b>UPTD Gudang Farmasi Surabaya</b><br>Titik Awal & Akhir",
            icon=folium.Icon(color='red', icon='home', prefix='fa')
        ).add_to(peta)

    algoritma_files = {
        "ga": {"nama": "Algoritma GA", "file_json": "../output_json/rute_ga.json", "file_ipynb": "ga.ipynb"},
        "ma": {"nama": "Algoritma MA", "file_json": "../output_json/rute_ma.json", "file_ipynb": "ma.ipynb"},
        "aco": {"nama": "Algoritma ACO", "file_json": "../output_json/aco_rute_per_wilayah.json", "file_ipynb": "sc_aco_revisimaps_fix.ipynb"},
        "pso": {"nama": "Algoritma PSO", "file_json": "../output_json/rute_dpso.json", "file_ipynb": "dpso.ipynb"}
    }

    jarak_optimasi_otomatis = {}
    print("Mengekstrak data jarak dari file .ipynb...")
    for id_algo, info in algoritma_files.items():
        jarak_optimasi_otomatis[id_algo] = ekstrak_jarak_dari_ipynb(info["file_ipynb"])
        print(f"Jarak {id_algo.upper()} berhasil dibaca: {jarak_optimasi_otomatis[id_algo]}")

    for id_algo, info in algoritma_files.items():
        filepath = info["file_json"]
        nama_algo = info["nama"]
        
        if not os.path.exists(filepath):
            continue
            
        with open(filepath, 'r') as f:
            data_rute = json.load(f)
            
        for wilayah, urutan_lokasi in data_rute.items():
            wilayah_lower_key = "lainnya"
            warna_rute = "#bdc3c7"
            
            for key, warna in warna_wilayah.items():
                if key in wilayah.lower():
                    wilayah_lower_key = key
                    warna_rute = warna
                    break
            
            class_identitas = f"algo-{id_algo} wil-{wilayah_lower_key}"

            for i in range(len(urutan_lokasi) - 1):
                lokasi_asal = urutan_lokasi[i].strip()
                lokasi_tujuan = urutan_lokasi[i+1].strip()
                
                if lokasi_asal in koordinat_lokasi and lokasi_tujuan in koordinat_lokasi:
                    coord_asal = koordinat_lokasi[lokasi_asal]
                    coord_tujuan = koordinat_lokasi[lokasi_tujuan]
                    
                    rute_nyata, jarak_m = get_real_route_osrm(coord_asal, coord_tujuan)
                    
                    folium.PolyLine(
                        locations=rute_nyata,
                        color=warna_rute,
                        weight=6, 
                        opacity=0.9,
                        className=class_identitas,
                        tooltip=f"<span style='color:{warna_rute}; font-weight:bold;'>{nama_algo} - {wilayah}</span><br>{lokasi_asal} ➔ {lokasi_tujuan}"
                    ).add_to(peta)
                    
                    if lokasi_tujuan != nama_gudang:
                        folium.CircleMarker(
                            location=coord_tujuan,
                            radius=6,
                            popup=f"<b>{lokasi_tujuan}</b><br>Klaster: {wilayah}",
                            color="white", 
                            weight=1.5,
                            fill=True,
                            fill_color=warna_rute,
                            fill_opacity=1,
                            className=class_identitas
                        ).add_to(peta)
                    
                    time.sleep(0.1) 

    jarak_js = json.dumps(jarak_optimasi_otomatis)

    custom_dashboard = f"""
    <style>
        @import url('https://fonts.googleapis.com/css2?family=Poppins:wght@400;600;700&display=swap');
        .custom-dashboard {{
            position: absolute; top: 20px; left: 20px; z-index: 9999;
            background: rgba(255, 255, 255, 0.95); backdrop-filter: blur(10px);
            padding: 20px; border-radius: 16px;
            box-shadow: 0px 10px 30px rgba(0, 0, 0, 0.1);
            font-family: 'Poppins', sans-serif; color: #2d3436;
            width: 320px; border: 1px solid rgba(255,255,255,0.5);
        }}
        .dash-title {{ font-size: 15px; font-weight: 700; margin: 0 0 15px 0; letter-spacing: 0.5px; text-transform: uppercase; color: #2d3436; border-bottom: 2px solid #f1f2f6; padding-bottom: 10px;}}
        .section-label {{ font-size: 11px; font-weight: 600; color: #636e72; margin-bottom: 8px; text-transform: uppercase; letter-spacing: 1px;}}
        
        .btn-container {{ display: flex; flex-wrap: wrap; gap: 8px; margin-bottom: 15px; }}
        .btn-pill {{
            padding: 6px 12px; border-radius: 20px; border: none; font-size: 12px; font-weight: 600;
            cursor: pointer; transition: all 0.3s ease; background: #f1f2f6; color: #636e72; font-family: 'Poppins', sans-serif;
        }}
        .btn-algo.active {{ background: #2d3436; color: #ffffff; box-shadow: 0 4px 10px rgba(45, 52, 54, 0.3); }}
        
        #btn-semua-wil {{ margin-bottom: 15px; width: 100%; transition: all 0.3s ease; }}

        .btn-wil.active-pusat {{ background: #f8a5c2; color: #fff; box-shadow: 0 4px 10px rgba(248, 165, 194, 0.4); }}
        .btn-wil.active-utara {{ background: #63cdda; color: #fff; box-shadow: 0 4px 10px rgba(99, 205, 218, 0.4); }}
        .btn-wil.active-selatan {{ background: #78e08f; color: #fff; box-shadow: 0 4px 10px rgba(120, 224, 143, 0.4); }}
        .btn-wil.active-timur {{ background: #cd84f1; color: #fff; box-shadow: 0 4px 10px rgba(205, 132, 241, 0.4); }}
        .btn-wil.active-barat {{ background: #f6b93b; color: #fff; box-shadow: 0 4px 10px rgba(246, 185, 59, 0.4); }}

        .distance-box {{ background: #f8f9fa; border-radius: 12px; padding: 15px; margin-top: 5px; text-align: center; border: 1px solid #e9ecef; }}
        .dist-label {{ font-size: 12px; color: #636e72; font-weight: 600; display: block; }}
        .dist-value {{ font-size: 28px; font-weight: 700; color: #2d3436; line-height: 1.2; display: inline-block; margin-top: 5px; }}
        .dist-unit {{ font-size: 14px; font-weight: 600; color: #b2bec3; }}

        .hide-ga .algo-ga {{ display: none !important; }}
        .hide-ma .algo-ma {{ display: none !important; }}
        .hide-aco .algo-aco {{ display: none !important; }}
        .hide-pso .algo-pso {{ display: none !important; }}
        
        .hide-pusat .wil-pusat {{ display: none !important; }}
        .hide-utara .wil-utara {{ display: none !important; }}
        .hide-selatan .wil-selatan {{ display: none !important; }}
        .hide-timur .wil-timur {{ display: none !important; }}
        .hide-barat .wil-barat {{ display: none !important; }}
    </style>

    <div class="custom-dashboard" id="mainDashboard">
        <h3 class="dash-title">📊 Rute Distribusi Farmasi</h3>
        <div class="section-label">Pilih Algoritma</div>
        <div class="btn-container">
            <button class="btn-pill btn-algo active" data-algo="ga">GA</button>
            <button class="btn-pill btn-algo" data-algo="ma">MA</button>
            <button class="btn-pill btn-algo" data-algo="aco">ACO</button>
            <button class="btn-pill btn-algo" data-algo="pso">PSO</button>
        </div>
        <div class="section-label">Filter Wilayah (Klaster)</div>
        <button class="btn-pill" id="btn-semua-wil"></button>
        <div class="btn-container">
            <button class="btn-pill btn-wil active-pusat" data-wil="pusat">Pusat</button>
            <button class="btn-pill btn-wil active-utara" data-wil="utara">Utara</button>
            <button class="btn-pill btn-wil active-selatan" data-wil="selatan">Selatan</button>
            <button class="btn-pill btn-wil active-timur" data-wil="timur">Timur</button>
            <button class="btn-pill btn-wil active-barat" data-wil="barat">Barat</button>
        </div>
        <div class="distance-box">
            <span class="dist-label">ESTIMASI JARAK OPTIMASI</span>
            <div><span class="dist-value" id="distDisplay">0.00</span> <span class="dist-unit">km</span></div>
        </div>
    </div>

    <script>
    setTimeout(function() {{
        const mapContainer = document.querySelector('.leaflet-container');
        const dataJarak = {jarak_js}; 
        
        const algoBtns = document.querySelectorAll('.btn-algo');
        const wilBtns = document.querySelectorAll('.btn-wil');
        const btnSemuaWil = document.getElementById('btn-semua-wil');
        const distDisplay = document.getElementById('distDisplay');

        let activeAlgo = "ga";
        let activeWils = new Set(["pusat", "utara", "selatan", "timur", "barat"]);

        function updateSelectAllBtn() {{
            if (activeWils.size > 0) {{
                btnSemuaWil.innerText = "Hapus Semua Pilihan";
                btnSemuaWil.style.background = "#ff7675"; 
                btnSemuaWil.style.color = "#ffffff";
                btnSemuaWil.style.boxShadow = "0 4px 10px rgba(255, 118, 117, 0.3)";
            }} else {{
                btnSemuaWil.innerText = "Pilih Semua Wilayah";
                btnSemuaWil.style.background = "#74b9ff"; 
                btnSemuaWil.style.color = "#ffffff";
                btnSemuaWil.style.boxShadow = "0 4px 10px rgba(116, 185, 255, 0.3)";
            }}
        }}

        function updateDashboard() {{
            ['ga', 'ma', 'aco', 'pso'].forEach(a => {{
                if(a === activeAlgo) mapContainer.classList.remove('hide-' + a);
                else mapContainer.classList.add('hide-' + a);
            }});
            ['pusat', 'utara', 'selatan', 'timur', 'barat'].forEach(w => {{
                if(activeWils.has(w)) mapContainer.classList.remove('hide-' + w);
                else mapContainer.classList.add('hide-' + w);
            }});

            let totalKm = 0;
            if(dataJarak[activeAlgo]) {{
                activeWils.forEach(w => {{
                    if(dataJarak[activeAlgo][w]) {{
                        totalKm += dataJarak[activeAlgo][w];
                    }}
                }});
            }}
            distDisplay.innerText = totalKm.toFixed(2);
        }}

        algoBtns.forEach(btn => {{
            btn.addEventListener('click', function() {{
                algoBtns.forEach(b => b.classList.remove('active'));
                this.classList.add('active');
                activeAlgo = this.getAttribute('data-algo');
                updateDashboard();
            }});
        }});

        wilBtns.forEach(btn => {{
            btn.addEventListener('click', function() {{
                const w = this.getAttribute('data-wil');
                const activeClass = 'active-' + w;
                if(activeWils.has(w)) {{
                    activeWils.delete(w);
                    this.classList.remove(activeClass);
                }} else {{
                    activeWils.add(w);
                    this.classList.add(activeClass);
                }}
                updateSelectAllBtn();
                updateDashboard();
            }});
        }});

        btnSemuaWil.addEventListener('click', function() {{
            if (activeWils.size > 0) {{
                activeWils.clear();
                wilBtns.forEach(btn => {{
                    btn.classList.remove('active-' + btn.getAttribute('data-wil'));
                }});
            }} else {{
                wilBtns.forEach(btn => {{
                    const w = btn.getAttribute('data-wil');
                    activeWils.add(w);
                    btn.classList.add('active-' + w);
                }});
            }}
            updateSelectAllBtn();
            updateDashboard();
        }});

        mapContainer.classList.add('hide-ma', 'hide-aco', 'hide-pso');
        updateSelectAllBtn();
        updateDashboard();

    }}, 1000);
    </script>
    """
    peta.get_root().html.add_child(folium.Element(custom_dashboard))

    os.makedirs("../output_json", exist_ok=True)
    output_html = "../output_json/visualisasi_perbandingan_rute.html"
    peta.save(output_html)
    return peta

peta_hasil = buat_peta_rute()
peta_hasil

Mengekstrak data jarak dari file .ipynb...
Jarak GA berhasil dibaca: {'selatan': 0, 'timur': 0, 'utara': 0, 'barat': 0, 'pusat': 0}
Jarak MA berhasil dibaca: {'selatan': 0, 'timur': 0, 'utara': 0, 'barat': 0, 'pusat': 0}
Jarak ACO berhasil dibaca: {'selatan': 35.41, 'timur': 28.24, 'utara': 34.85, 'barat': 52.35, 'pusat': 26.11}
Jarak PSO berhasil dibaca: {'selatan': 0, 'timur': 0, 'utara': 0, 'barat': 0, 'pusat': 0}


KeyboardInterrupt: 